# Microsoft Agent Framework Multi-Agent Harness

This notebook studies one production-style incident workflow using Microsoft Agent Framework (MAF) with Ollama Cloud as the model backend.

The workflow uses multiple specialist agents and a harnessed synthesizer:

1. evidence agent reads the incident and logs
2. runbook agent extracts approved actions and forbidden actions
3. memory agent extracts prior incident lessons
4. harness synthesizer composes the final JSON plan using MAF `create_harness_agent`
5. the final output is evaluated by deterministic rules and a qualitative LLM critic

The harness mechanics come from MAF. The scorer and critic sit outside the workflow so the result can be measured independently.

This notebook uses the direct Ollama Cloud API path. It does not require `ollama serve`, `ollama pull`, or a local model download.


## 1. Clone And Install

Run this in a fresh Colab runtime. MAF is installed separately from the base project dependencies so any package issue is easy to isolate.

This notebook uses Ollama Cloud through its OpenAI-compatible `/v1/` endpoint. No local Ollama daemon or model pull is needed.


In [ ]:
REPO_URL = "https://github.com/narendra-devireddy/ollama-harness-engineering-demo.git"
REPO_DIR = "/content/ollama-harness-engineering-demo"

from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone "$REPO_URL" "$REPO_DIR"

%cd "$REPO_DIR"

!python -m pip install -q pydantic pyyaml rich typer ollama openai
!python -m pip install -q --pre agent-framework agent-framework-openai agent-framework-orchestrations

# The repo is imported through sys.path below, so this notebook avoids `pip install -e .`
# and does not install Strands-specific packages.


## 2. Ollama Cloud Key And Model Probe

This notebook talks directly to Ollama Cloud through the OpenAI-compatible endpoint `https://ollama.com/v1/`. Add `OLLAMA_API_KEY` in Colab Secrets if possible; otherwise this cell will ask for it without printing the value.

The probe below selects a model that is suitable for a harnessed multi-agent workflow. A tiny smoke test is not enough; the model must tolerate the larger instruction envelope that `create_harness_agent` adds.


In [ ]:
import os
from getpass import getpass
from openai import OpenAI

OLLAMA_CLOUD_HOST = "https://ollama.com"
OLLAMA_OPENAI_BASE_URL = "https://ollama.com/v1/"

# Use the DeepSeek Flash Cloud route that supports this workflow reliably.
# A plain completion probe is not enough: the MAF harness also needs native tool calls.
PREFERRED_MAF_MODELS = ["deepseek-v4-flash:0731", "deepseek-v4-flash:cloud"]
SUMMARY_MODEL = "deepseek-v4-flash:0731"

try:
    from google.colab import userdata
    secret_value = userdata.get("OLLAMA_API_KEY")
    if secret_value:
        os.environ["OLLAMA_API_KEY"] = secret_value
except Exception:
    pass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Paste Ollama Cloud API key: ")

cloud_probe = OpenAI(
    base_url=OLLAMA_OPENAI_BASE_URL,
    api_key=os.environ["OLLAMA_API_KEY"],
)

print("Ollama Cloud OpenAI-compatible endpoint:", OLLAMA_OPENAI_BASE_URL)

visible_models = []
try:
    models = cloud_probe.models.list()
    visible_models = [model.id for model in models.data]
    if visible_models:
        print("Visible cloud models:", ", ".join(visible_models[:20]))
except Exception as exc:
    print("Model listing skipped:", type(exc).__name__, str(exc))


def probe_model(model_name: str, max_tokens: int = 512) -> bool:
    try:
        response = cloud_probe.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Reply with exactly: cloud model ok"}],
            max_tokens=max_tokens,
            temperature=0,
        )
        text = response.choices[0].message.content.strip()
        print(f"{model_name}: ok -> {text[:80]}")
        return True
    except Exception as exc:
        print(f"{model_name}: failed -> {type(exc).__name__}: {exc}")
        return False

MAF_MODEL = None
for candidate in PREFERRED_MAF_MODELS:
    # If the model list is available, skip names that are definitely absent.
    if visible_models and candidate not in visible_models:
        continue
    if probe_model(candidate):
        MAF_MODEL = candidate
        break

if MAF_MODEL is None:
    raise RuntimeError("No preferred Ollama Cloud model passed the probe. Check the visible model names above.")

print("Selected MAF model:", MAF_MODEL)
print("Selected critique model:", SUMMARY_MODEL)


## Optional Local Ollama Shutdown

This notebook does not start a local Ollama server. If a previous notebook left `ollama serve` running in the Colab terminal, use this optional cleanup cell.


In [ ]:
STOP_LOCAL_OLLAMA_SERVER = False

if STOP_LOCAL_OLLAMA_SERVER:
    print("Stopping local Ollama processes from an earlier run...")
    !pkill -f "ollama serve" || true
    !pkill -f ollama || true
else:
    print("No local Ollama process is needed for this notebook.")


## 3. Import Scenario And Evaluation Tools

The incident, runbook, prior memory, scorer, and qualitative critic are imported from the repository. This keeps the MAF run comparable with the other harness approaches.

In [ ]:
import json
import os
import sys
from pathlib import Path
from pprint import pprint
from typing import Any
from IPython.display import Markdown, display

REPO_DIR = Path(os.environ.get("HARNESS_DEMO_REPO_DIR", "/content/ollama-harness-engineering-demo"))
SRC_DIR = REPO_DIR / "src"
for path in (str(SRC_DIR), str(REPO_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

from harness_demo.domain import Lane
from harness_demo.live import score_freeform_answer
from harness_demo.rules import evaluate_rules
from harness_demo.scenarios import load_incident_scenario
from harness_demo.colab_display import (
    render_executive_findings_markdown,
    render_memory_markdown,
    render_model_output_html,
    render_quality_gate_markdown,
    render_rule_findings_markdown,
)
from harness_demo.summarizer import critique_groundedness_with_ollama, summarize_findings_with_ollama

scenario = load_incident_scenario("incident-response")
print("Incident:")
pprint(scenario.incident)
print("\nQuality contract:")
pprint(scenario.expected)

## 4. Use A Model That Fits The Harness Envelope

A plain model call and a harnessed multi-agent call are not the same request. `create_harness_agent` adds system instructions, todo/mode guidance, loop guidance, and optional provider tools. The selected MAF model must have enough effective context for that full envelope.

For this notebook, prefer a high-context Ollama Cloud model such as `deepseek-v4-flash` when it is available. The scorer still evaluates the final artifact against the same deterministic quality contract.


In [ ]:
print("Selected MAF model:", MAF_MODEL)
print("Selected critique model:", SUMMARY_MODEL)
print("Model backend:", OLLAMA_OPENAI_BASE_URL)


## 5. Configure Model Client

MAF uses its Chat Completions client pointed directly at Ollama Cloud's OpenAI-compatible endpoint. This route avoids the local Ollama daemon and avoids Responses-style service-managed conversation state.

The same cloud route is also wrapped for the qualitative critique cells later in the notebook.


In [ ]:
from openai import OpenAI
from agent_framework.openai import OpenAIChatCompletionClient

# Ollama Cloud currently supports the OpenAI Chat Completions route for this
# integration. OpenAIChatClient may select /v1/responses, which returns a
# provider 500 from Ollama Cloud.
client = OpenAIChatCompletionClient(
    base_url=OLLAMA_OPENAI_BASE_URL,
    api_key=os.environ["OLLAMA_API_KEY"],
    model=MAF_MODEL,
)

class NotebookOllamaCloudModel:
    def __init__(self, model_name, base_url=OLLAMA_OPENAI_BASE_URL, timeout_seconds=90):
        self.model_name = model_name
        self._client = OpenAI(
            base_url=base_url,
            api_key=os.environ["OLLAMA_API_KEY"],
            timeout=timeout_seconds,
        )

    def chat(self, messages):
        response = self._client.chat.completions.create(
            model=self.model_name,
            messages=messages,
            max_tokens=1200,
            temperature=0,
            stream=False,
        )
        return response.choices[0].message.content

SUMMARY_CHAT_MODEL = NotebookOllamaCloudModel(SUMMARY_MODEL)

# MAF's todo loop requires real OpenAI-format tool calls. Check that capability
# directly before enabling the loop, rather than discovering incompatibility
# after several failed harness iterations.
def probe_native_tool_calls(model_name: str) -> bool:
    def record_incident_fact(fact: str) -> str:
        return fact

    try:
        response = cloud_probe.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Call record_incident_fact with fact='tool probe ok'."}],
            tools=[{"type": "function", "function": {"name": "record_incident_fact", "description": "Record one fact.", "parameters": {"type": "object", "properties": {"fact": {"type": "string"}}, "required": ["fact"]}}}],
            tool_choice="required",
            max_tokens=128,
            temperature=0,
        )
        calls = getattr(response.choices[0].message, "tool_calls", None) or []
        ok = bool(calls and calls[0].function.name == "record_incident_fact")
        print("Native tool-call probe:", "passed" if ok else "failed")
        if not ok:
            print("Probe content:", (response.choices[0].message.content or "")[:240])
        return ok
    except Exception as exc:
        print("Native tool-call probe failed:", type(exc).__name__, str(exc))
        return False

MAF_NATIVE_TOOL_CALLS = probe_native_tool_calls(MAF_MODEL)
print("MAF todo loop enabled:", MAF_NATIVE_TOOL_CALLS)

print("MAF provider: agent_framework.openai.OpenAIChatCompletionClient")
print("MAF model:", MAF_MODEL)
print("Critique model:", SUMMARY_MODEL)
print("Endpoint:", OLLAMA_OPENAI_BASE_URL)


## 6. Build The MAF Multi-Agent Harness

Each workflow role is created with MAF `create_harness_agent`, not the plain `Agent` API. That means every specialist gets the same production harness substrate: todo planning, mode tracking, compaction limits, persistence, and bounded loop settings when supported by the installed framework version. Hosted web search is disabled because this incident exercise is fully source-grounded and Ollama Cloud's OpenAI-compatible endpoint rejects MAF's default hosted `web_search` tool.

The agents still have different responsibilities:

- evidence harness agent: recover only source-backed log evidence
- runbook harness agent: recover approved actions and forbidden actions
- memory harness agent: recover relevant prior-incident lessons
- synthesizer harness agent: compose the final structured incident plan

MAF `SequentialBuilder` then passes the specialist outputs forward and surfaces intermediate outputs where supported.


The agent factory sets both harness token budgets and the chat-completion `max_tokens` option. These are separate controls: the first helps MAF reason about compaction, while the second controls how much completion space the Ollama Cloud endpoint reserves.


In [ ]:
import inspect
from agent_framework import AgentResponse, create_harness_agent, todos_remaining, todos_remaining_message
from agent_framework.orchestrations import SequentialBuilder


def as_text(response: Any) -> str:
    """Best-effort text extraction across MAF response/event shapes."""
    if response is None:
        return ""
    if isinstance(response, str):
        return response
    if isinstance(response, (list, tuple, set)):
        return "\n\n".join(part for item in response if (part := as_text(item)))
    if isinstance(response, dict):
        priority_keys = ("text", "content", "contents", "message", "messages", "value", "data", "output")
        parts = [as_text(response.get(key)) for key in priority_keys if key in response]
        parts = [part for part in parts if part]
        return "\n\n".join(parts)

    direct_parts = []
    for attr in ("text", "content", "contents", "value", "message", "messages", "data", "output"):
        if hasattr(response, attr):
            try:
                value = getattr(response, attr)
            except Exception:
                continue
            text = as_text(value)
            if text:
                direct_parts.append(text)
    if direct_parts:
        return "\n\n".join(direct_parts)

    # Some SDK objects expose useful data through model_dump/dict/json.
    for method_name in ("model_dump", "dict"):
        method = getattr(response, method_name, None)
        if callable(method):
            try:
                text = as_text(method())
                if text:
                    return text
            except Exception:
                pass

    rendered = str(response)
    if rendered and rendered != object.__repr__(response):
        return rendered
    return ""


def describe_response_shape(response: Any) -> dict:
    """Small debug summary for provider-specific response shapes."""
    info = {"type": type(response).__name__}
    for attr in ("text", "content", "contents", "value", "message", "messages", "data", "output"):
        try:
            if hasattr(response, attr):
                value = getattr(response, attr)
                info[attr] = type(value).__name__
                preview = as_text(value)[:180]
                if preview:
                    info[attr + "_preview"] = preview
        except Exception as exc:
            info[attr] = f"unreadable: {type(exc).__name__}"
    try:
        rendered = str(response)
        if rendered:
            info["str_preview"] = rendered[:300]
    except Exception:
        pass
    return info

harness_signature = inspect.signature(create_harness_agent)
supported_options = set(harness_signature.parameters)
print("create_harness_agent accepted options:")
print(", ".join(sorted(supported_options)))



def make_harness_agent(name: str, description: str, instructions: str, max_iterations: int = 2):
    """Create one MAF harness agent while only passing options supported by the installed package."""
    max_context_tokens = 100_000
    max_output_tokens = 4_096

    kwargs = {"client": client}
    optional_kwargs = {
        "name": name,
        "description": description,
        "agent_instructions": instructions,
        "max_context_window_tokens": max_context_tokens,
        "max_output_tokens": max_output_tokens,
        # This is the actual Chat Completions API output cap. Without it, the provider may reserve too much response space.
        "default_options": {"max_tokens": max_output_tokens, "temperature": 0},
        "disable_web_search": True,
        "disable_file_memory": True,
    }
    if MAF_NATIVE_TOOL_CALLS:
        optional_kwargs.update({
            "loop_should_continue": todos_remaining(looping_modes=["execute"]),
            "loop_next_message": todos_remaining_message,
            "loop_max_iterations": max_iterations,
        })
    else:
        # Do not expose MAF todo/mode tools when the provider fails the
        # native tool-call probe. The other harness components remain active.
        optional_kwargs.update({"disable_todo": True, "disable_mode": True})

    unsupported = []
    for key, value in optional_kwargs.items():
        if key in supported_options:
            kwargs[key] = value
        else:
            unsupported.append(key)

    if unsupported:
        print(f"{name}: skipped unsupported create_harness_agent options:", ", ".join(unsupported))
    print(f"{name}: active harness options:", ", ".join(key for key in kwargs if key != "client"))
    return create_harness_agent(**kwargs), {key: str(value) for key, value in kwargs.items() if key != "client"}


loop_guidance = (
    "Use the built-in todo tools and finish all open todos."
    if MAF_NATIVE_TOOL_CALLS
    else "Work in one pass and do not call tools; return the requested text directly."
)
evidence_instructions = f"""Extract only log/ticket evidence. {loop_guidance} No mitigations. No invented tools, owners, dashboards, or metrics. Return EVIDENCE_FINDINGS bullets only."""

runbook_instructions = f"""Extract approved actions, thresholds, rollback guidance, and forbidden actions from the runbook. {loop_guidance} Treat runbook as policy. Return RUNBOOK_FINDINGS bullets only."""

memory_instructions = f"""Extract only relevant prior-incident lessons. {loop_guidance} Do not add policy. Return MEMORY_FINDINGS bullets only."""

synthesizer_instructions = f"""Compose final incident JSON from specialist outputs and sources. {loop_guidance} Include required fields: likely_cause, evidence, safe_next_action, rollback_plan, customer_impact, open_questions. Follow runbook. No forbidden actions. Return JSON only."""

agent_specs = [
    ("evidence-harness-agent", "MAF harness agent that extracts log-backed evidence.", evidence_instructions, 2),
    ("runbook-harness-agent", "MAF harness agent that extracts approved runbook actions and constraints.", runbook_instructions, 2),
    ("memory-harness-agent", "MAF harness agent that extracts prior incident lessons.", memory_instructions, 2),
    ("synthesizer-harness-agent", "MAF harness agent that composes the final incident plan.", synthesizer_instructions, 3),
]

created_agents = []
harness_options_by_agent = {}
for name, description, instructions, max_iterations in agent_specs:
    agent, options = make_harness_agent(name, description, instructions, max_iterations=max_iterations)
    created_agents.append(agent)
    harness_options_by_agent[name] = options

evidence_agent, runbook_agent, memory_agent, synthesizer_agent = created_agents
participants = [evidence_agent, runbook_agent, memory_agent, synthesizer_agent]

try:
    maf_workflow = SequentialBuilder(
        participants=participants,
        intermediate_output_from=[evidence_agent, runbook_agent, memory_agent],
        output_from=[synthesizer_agent],
    ).build()
    output_mode = "intermediate_output_from + output_from"
except TypeError:
    maf_workflow = SequentialBuilder(
        participants=participants,
        intermediate_output_from=[evidence_agent, runbook_agent, memory_agent],
    ).build()
    output_mode = "intermediate_output_from + default final output"

print("MAF harness agents:")
for agent in participants:
    print("-", getattr(agent, "name", type(agent).__name__))
print("\nEvery workflow role was created through create_harness_agent.")
print("Workflow output mode:", output_mode)
print("\nHarness options by agent:")
print(json.dumps(harness_options_by_agent, indent=2))


## 7. Run The Workflow With Intermediate Output

The default run is non-streaming so the final response remains clean and readable. MAF intermediate outputs are collected from the workflow result where the installed version supports them.

Set `STREAM_MAF_WORKFLOW = True` only when you want to inspect raw stream events. Stream chunks are useful for observability, but they are not used as the final artifact for scoring.

In [ ]:
import asyncio
import re

def expected_items(name: str):
    expected = scenario.expected
    if isinstance(expected, dict):
        return expected.get(name, []) or []
    return getattr(expected, name, []) or []

maf_task = f"""
INCIDENT {json.dumps(scenario.incident, separators=(",", ":"))}
LOGS {scenario.logs}
RUNBOOK {scenario.runbook}
MEMORY {scenario.prior_memory}
REQUIRED_FIELDS likely_cause,evidence,safe_next_action,rollback_plan,customer_impact,open_questions
REQUIRED_EVIDENCE {' | '.join(expected_items('required_evidence'))}
REQUIRED_RUNBOOK {' | '.join(expected_items('required_runbook_steps'))}
FORBIDDEN {' | '.join(expected_items('forbidden_actions'))}
Return the final plan as compact JSON.
""".strip()

STREAM_MAF_WORKFLOW = False
MAF_RUN_TIMEOUT_SECONDS = 240

maf_stream_events = []
maf_intermediate_parts = []
maf_output = ""


def clean_model_text(text: str) -> str:
    text = text.replace("\r\n", "\n")
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def readable_maf_content(content) -> str:
    if content is None or isinstance(content, (bool, int, float)):
        return ""
    if isinstance(content, str):
        return content.strip()
    content_type = str(getattr(content, "type", "")).lower()
    if any(marker in content_type for marker in ("reasoning", "function_call", "function_result", "tool_call", "tool_result")):
        return ""
    text = getattr(content, "text", None)
    if isinstance(text, str) and text.strip():
        return text.strip()
    return ""

def readable_maf_output(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (list, tuple, set)):
        parts = [readable_maf_output(item) for item in value]
        return "\n\n".join(part for part in parts if part)
    if isinstance(value, dict):
        for key in ("text", "content", "contents", "messages", "output"):
            if key in value:
                text = readable_maf_output(value[key])
                if text:
                    return text
        return ""
    contents = getattr(value, "contents", None)
    if contents is not None:
        text = readable_maf_output(contents)
        if text:
            return text
    text = readable_maf_content(value)
    if text:
        return text
    messages = getattr(value, "messages", None)
    if messages is not None:
        return readable_maf_output(messages)
    return ""


async def run_maf_non_streaming():
    result = await asyncio.wait_for(maf_workflow.run(maf_task), timeout=MAF_RUN_TIMEOUT_SECONDS)
    outputs = list(result.get_outputs())
    final_parts = [readable_maf_output(output) for output in outputs]
    try:
        intermediate_outputs = list(result.get_intermediate_outputs())
        intermediate_parts = [readable_maf_output(output) for output in intermediate_outputs]
    except Exception:
        intermediate_outputs = []
        intermediate_parts = []

    if not any(part.strip() for part in final_parts + intermediate_parts):
        print("MAF returned response objects, but no text was found by the first extractor pass.")
        print("Final output object shapes:")
        print(json.dumps([describe_response_shape(output) for output in outputs], indent=2))
        print("Intermediate output object shapes:")
        print(json.dumps([describe_response_shape(output) for output in intermediate_outputs], indent=2))
        whole_result_text = readable_maf_output(result)
        if whole_result_text:
            final_parts = [whole_result_text]
    return final_parts, intermediate_parts


async def inspect_maf_stream_only():
    print("Inspecting raw MAF stream events. These chunks are not used for final scoring.")
    async for event in maf_workflow.run(maf_task, stream=True):
        event_type = getattr(event, "type", type(event).__name__)
        data = getattr(event, "data", event)
        text = getattr(data, "text", "") or ""
        author = getattr(data, "author_name", "") or getattr(data, "name", "") or ""
        maf_stream_events.append({
            "index": len(maf_stream_events) + 1,
            "event_type": event_type,
            "author": author,
            "has_text": bool(text.strip()),
            "preview": clean_model_text(text)[:220],
        })


if STREAM_MAF_WORKFLOW:
    try:
        await asyncio.wait_for(inspect_maf_stream_only(), timeout=MAF_RUN_TIMEOUT_SECONDS)
        display(Markdown("## MAF Raw Stream Event Table"))
        try:
            import pandas as pd
            display(pd.DataFrame(maf_stream_events))
        except Exception:
            print(json.dumps(maf_stream_events, indent=2))
    except Exception as exc:
        print("Stream inspection skipped/failed:", type(exc).__name__, str(exc))

print("Running MAF workflow for clean final output...")
try:
    maf_final_parts, maf_intermediate_parts = await run_maf_non_streaming()
    maf_output = clean_model_text("\n\n".join(part for part in maf_final_parts if part and part.strip()))
    if not maf_output:
        maf_output = clean_model_text("\n\n".join(part for part in maf_intermediate_parts if part and part.strip()))
    print("MAF workflow completed.")
    print("Final output characters:", len(maf_output))
except asyncio.TimeoutError:
    maf_output = ""
    display(Markdown("## MAF workflow timed out"))
    print(f"No response after {MAF_RUN_TIMEOUT_SECONDS} seconds. Check Ollama server/model connectivity and rerun this cell.")
except Exception as exc:
    maf_output = ""
    display(Markdown("## MAF workflow did not complete"))
    print(type(exc).__name__, str(exc))

if maf_intermediate_parts:
    display(Markdown("## MAF Intermediate Outputs"))
    for index, part in enumerate(maf_intermediate_parts, start=1):
        display(Markdown(render_model_output_html(f"Intermediate output {index}", clean_model_text(part))))


## 8. Evaluate The MAF Output

The MAF workflow produced the answer. The independent evaluation layer checks whether that answer is grounded, runbook-aligned, safe, memory-aware, and complete.

In [ ]:
raw_maf_result = score_freeform_answer(
    scenario=scenario,
    answer=maf_output,
    lane=Lane.MICROSOFT_AGENT_FRAMEWORK,
    title=f"Microsoft Agent Framework multi-agent harness ({MAF_MODEL})",
    takeaway=(
        "MAF provides the multi-agent orchestration and harnessed synthesizer. "
        "The external evaluator measures the generated artifact against the incident quality contract."
    ),
    used_harness_memory=True,
)

# Only the synthesized final plan is eligible for scoring and critique.
# Specialist/intermediate outputs remain visible above for observability.
canonical_maf_output = json.dumps(raw_maf_result.memory.final_plan, indent=2, default=str)
maf_result = score_freeform_answer(
    scenario=scenario,
    answer=canonical_maf_output,
    lane=Lane.MICROSOFT_AGENT_FRAMEWORK,
    title=f"Microsoft Agent Framework final plan ({MAF_MODEL})",
    takeaway=(
        "MAF orchestrates the agents; the canonical synthesized plan is the artifact evaluated by the harness."
    ),
    used_harness_memory=True,
)
print("Evaluation artifact: synthesized final plan only")

display(Markdown(render_quality_gate_markdown(maf_result)))
display(Markdown(render_executive_findings_markdown(maf_result)))
display(Markdown(render_rule_findings_markdown(scenario, maf_result)))
display(Markdown(render_memory_markdown(maf_result.memory)))
display(Markdown(render_model_output_html("Actual MAF final plan", canonical_maf_output)))

## 9. Qualitative LLM Critique

The deterministic scorer makes the pass/fail decision. The LLM critic is used only to explain subtle unsupported claims, overconfidence, or wording risks that string rules may miss.

In [ ]:
try:
    critique = critique_groundedness_with_ollama(
        scenario,
        maf_result,
        model_name=SUMMARY_MODEL,
        model=SUMMARY_CHAT_MODEL,
    )
    display(Markdown("## Qualitative Groundedness Critique"))
    display(Markdown(critique))
except Exception as exc:
    print("No qualitative critique generated:", type(exc).__name__, str(exc))

try:
    summary = summarize_findings_with_ollama(
        scenario,
        maf_result,
        evaluate_rules(scenario, maf_result),
        model_name=SUMMARY_MODEL,
        model=SUMMARY_CHAT_MODEL,
    )
    display(Markdown("## Plain-Language Summary"))
    display(Markdown(summary))
except Exception as exc:
    print("No plain-language summary generated:", type(exc).__name__, str(exc))

## 10. What To Notice

- The first three agents narrow the context before synthesis.
- MAF orchestration passes each specialist output forward.
- The synthesizer is a MAF harness agent, so planning/todo state, mode tracking, compaction limits, and built-in loop behavior are configured through the framework.
- Intermediate outputs make the workflow inspectable without writing a separate custom retry engine.
- Evaluation remains independent, so the framework is measured against the same quality contract as the other approaches.